# DW-1: внешние эмбеддинги топ-50
Запускайте в Colab; локально torch не устанавливается. Загрузите external_bundle.zip,
созданный `python -m dw1.campaigns.review`. Допускается только утверждённый публичный
корпус с указанной лицензией. Ноутбук не производит ручную разметку и не вычисляет F1.
Модель может обрезать длинные тексты: max_seq_length фиксируется в metadata.


In [ ]:
%pip -q install sentence-transformers==3.4.1 numpy==1.26.4 huggingface-hub==0.28.1 transformers==4.48.3


In [ ]:
import json
import os
import subprocess
import sys
import zipfile
from pathlib import Path

from google.colab import files

uploaded = files.upload()
assert set(uploaded) == {"external_bundle.zip"}, "Выберите только external_bundle.zip"
root = Path("/content/dw1-campaigns")
root.mkdir(exist_ok=True)
with zipfile.ZipFile("external_bundle.zip") as bundle:
    # Reject traversal paths before extraction.
    for name in bundle.namelist():
        target = (root / name).resolve()
        assert target.is_relative_to(root.resolve()), "Unsafe archive path"
    bundle.extractall(root)
manifest = json.loads((root / "manifest.json").read_text())
assert manifest["provenance"]["external_allowed"] is True
print(manifest["provenance"])
print("Документов:", manifest["n_documents"], "кластеров:", manifest["top_n"])


In [ ]:
from huggingface_hub import model_info

model_id = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
# Resolve once; the exporter records and uses this exact commit, not moving main.
revision = model_info(model_id).sha
print("Pinned model revision:", revision)
env = {**os.environ, "PYTHONPATH": str(root / "src")}
subprocess.run([sys.executable, str(root / "export_embeddings.py"),
    "--documents", str(root / "documents.jsonl"),
    "--manifest", str(root / "manifest.json"),
    "--out", str(root / "embeddings.npz"),
    "--model", model_id, "--revision", revision,
    "--public-data-approved"], env=env, check=True)
files.download(str(root / "embeddings.npz"))


Верните embeddings.npz в data/runs/sprint3/campaigns/review/.
Параллельно каждый из 2–3 проверяющих заполняет review.html и сохраняет свой JSONL.
После ручного согласования групп запускается make campaign-eval.
